# 🔥 Aumento de complejidad en PySpark con múltiples lecturas y uniones
Este notebook demuestra cómo la complejidad del plan de ejecución de Spark aumenta cuando:
- Se generan múltiples versiones de un Parquet.
- Se leen y unen en un bucle.
- Se realiza una agregación compleja sobre el dataset combinado.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Crear sesión Spark
spark = SparkSession.builder.appName('ComplejidadUnionParquet').getOrCreate()

# Ruta base del parquet original
base_path = '/var/snp-dwh/sftp/output/habitantes_procesado.parquet'

# Leer parquet base
df_base = spark.read.parquet(base_path)
df_base.show(5)

25/08/03 19:25:52 WARN Utils: Your hostname, testnode1 resolves to a loopback address: 127.0.1.1; using 192.168.1.144 instead (on interface eth0)
25/08/03 19:25:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/03 19:25:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+----------------+----------------+-------------------+
|nombre_provincia|total_habitantes|promedio_habitantes|
+----------------+----------------+-------------------+
|          GUAYAS|        28438626|          189590.84|
|       PICHINCHA|        19633590|         409033.125|
|          MANABI|        10196604|            77247.0|
|        LOS RIOS|         5811960|  74512.30769230769|
|           AZUAY|         4987320| 55414.666666666664|
+----------------+----------------+-------------------+
only showing top 5 rows



In [9]:
df_base.count()

24

## 1️⃣ Generar múltiples versiones del Parquet para simular datos de diferentes fuentes

In [2]:
# Crear múltiples versiones con ligeras modificaciones y guardarlas en subcarpetas
for i in range(1, 4):
    df_mod = df_base.withColumn('total_habitantes', F.col('total_habitantes') * (1 + i * 0.02)) \
                    .withColumn('version', F.lit(f'v{i}'))
    output_version_path = f'/var/snp-dwh/sftp/output/version_{i}'
    df_mod.write.mode('overwrite').parquet(output_version_path)

print('✅ Se generaron versiones v1, v2 y v3 del parquet')

✅ Se generaron versiones v1, v2 y v3 del parquet


## 2️⃣ Leer todas las versiones y unirlas en un bucle

In [8]:
paths = [f'/var/snp-dwh/sftp/output/version_{i}' for i in range(1, 4)]

# Leer el primer parquet como base
df_union = spark.read.parquet(paths[0])

# Unir el resto en un bucle
for p in paths[1:]:
    df_next = spark.read.parquet(p)
    df_union = df_union.unionByName(df_next)

print('✅ Unión de todas las versiones completada')
df_union.show(10)

✅ Unión de todas las versiones completada
+----------------+----------------+-------------------+-------+
|nombre_provincia|total_habitantes|promedio_habitantes|version|
+----------------+----------------+-------------------+-------+
|          GUAYAS|   2.900739852E7|          189590.84|     v1|
|       PICHINCHA|    2.00262618E7|         409033.125|     v1|
|          MANABI|   1.040053608E7|            77247.0|     v1|
|        LOS RIOS|       5928199.2|  74512.30769230769|     v1|
|           AZUAY|       5087066.4| 55414.666666666664|     v1|
|          EL ORO|      4581597.24| 53473.357142857145|     v1|
|      ESMERALDAS|      3681951.12|  85946.57142857143|     v1|
|      TUNGURAHUA|      3543981.84| 64342.444444444445|     v1|
|   SANTO DOMINGO|      3203966.88|           261762.0|     v1|
|            LOJA|      3044320.56|          31089.875|     v1|
+----------------+----------------+-------------------+-------+
only showing top 10 rows



## 3️⃣ Operación adicional: agregación compleja tras la unión

In [4]:
df_complex = df_union.groupBy('nombre_provincia') \
    .agg(
        F.sum('total_habitantes').alias('habitantes_acumulados'),
        F.avg('total_habitantes').alias('habitantes_promedio'),
        F.countDistinct('version').alias('fuentes')
    ) \
    .orderBy(F.desc('habitantes_acumulados'))

df_complex.show()

+----------------+---------------------+--------------------+-------+
|nombre_provincia|habitantes_acumulados| habitantes_promedio|fuentes|
+----------------+---------------------+--------------------+-------+
|          GUAYAS|        8.872851312E7|2.9576171040000003E7|      3|
|       PICHINCHA|  6.125680080000001E7|2.0418933600000005E7|      3|
|          MANABI| 3.1813404480000004E7|1.0604468160000002E7|      3|
|        LOS RIOS| 1.8133315200000003E7|   6044438.400000001|      3|
|           AZUAY| 1.5560438399999999E7|           5186812.8|      3|
|          EL ORO| 1.4014297440000001E7|          4671432.48|      3|
|      ESMERALDAS|        1.126243872E7|          3754146.24|      3|
|      TUNGURAHUA|        1.084041504E7|  3613471.6799999997|      3|
|   SANTO DOMINGO|    9800369.280000001|  3266789.7600000002|      3|
|            LOJA|           9312039.36|  3104013.1199999996|      3|
|        IMBABURA|            9248335.2|           3082778.4|      3|
|        COTOPAXI|  

## 4️⃣ Visualización de la complejidad del plan de ejecución

In [5]:
# Mostrar el plan de ejecución para observar cómo Spark maneja múltiples lecturas y uniones
df_complex.explain(True)

== Parsed Logical Plan ==
'Sort ['habitantes_acumulados DESC NULLS LAST], true
+- Aggregate [nombre_provincia#59], [nombre_provincia#59, sum(total_habitantes#60) AS habitantes_acumulados#123, avg(total_habitantes#60) AS habitantes_promedio#125, count(distinct version#62) AS fuentes#126L]
   +- Union false, false
      :- Relation [nombre_provincia#59,total_habitantes#60,promedio_habitantes#61,version#62] parquet
      :- Project [nombre_provincia#67, total_habitantes#68, promedio_habitantes#69, version#70]
      :  +- Relation [nombre_provincia#67,total_habitantes#68,promedio_habitantes#69,version#70] parquet
      +- Project [nombre_provincia#80, total_habitantes#81, promedio_habitantes#82, version#83]
         +- Relation [nombre_provincia#80,total_habitantes#81,promedio_habitantes#82,version#83] parquet

== Analyzed Logical Plan ==
nombre_provincia: string, habitantes_acumulados: double, habitantes_promedio: double, fuentes: bigint
Sort [habitantes_acumulados#123 DESC NULLS LAST], t